# Notebook 10 — Final benchmark and recommendation

Evaluate the accepted production regex, fitted word + character TF-IDF logistic regression, and **Qwen/Qwen3.6-35B-A3B-FP8** against the same approved reference.

**Scope:** 121 eligible event groups from the original 123-group holdout (149 reviewed dated occurrences). Two confirmed duplicate events already present in training/validation are excluded before predictions. Five rubric-v3 tag additions were approved before this run. Original records and labels remain preserved privately. This was a focused rubric update, not a complete independent second labeling pass; retained reviewer uncertainty still limits the answer key.

The prompt, fitted model, rules, 0.30 classifier cutoff and output policies were selected using development data. They are fixed here. All primary results use at most two predicted tags; reference tags are not truncated. A separate view retains exact ties for the scored models.

**Run:** default execution replays saved results. To make missing Hetzner requests, set `NOTEBOOK10_RUN_LIVE=true` in the local `.env` or kernel environment, then Run All. Only the already-authorized free-tier account is supported. Do not change settings after viewing final results. Cached results preserve original request time; replay speed is not inference speed.

In [ ]:
import json
import os
import re
import sys
import unicodedata
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from dotenv import dotenv_values
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "src"))
from notebooks.tools import file_sha256, percent, show_table, verify_sha256
from notebooks.tools.hosted import chat_request
from demo_logic.evaluation import multilabel_metrics
from demo_logic.notebook_support import save_private_json
from demo_logic.taxonomy import topic_tags

folder = root / "reports/final-benchmark/v1"
manifest = json.loads((folder / "manifest.json").read_text())
verify_sha256(folder / "candidates.json", manifest["candidate_contract_sha256"])
verify_sha256(folder / "reference.json", manifest["reference_sha256"])
contract = json.loads((folder / "candidates.json").read_text())
reference = json.loads((folder / "reference.json").read_text())
if reference["approval_status"] != "approved" or not manifest["final_labels_approved"]:
    raise ValueError("The final reference must be approved before inference")
for path, expected in contract["input_fingerprints"].items():
    verify_sha256(root / path, expected)
verify_sha256(root / "data/benchmark/final-review-queue.csv", reference["source_csv_sha256"])
verify_sha256(root / "docs/tagging-rubric.md", manifest["rubric_sha256"])
snapshot = {
    row["external_id"]: row
    for row in map(json.loads, (root / "data/dataset.jsonl").read_text().splitlines())
}
selection = json.loads((root / "data/benchmark/event-type-representatives-v1.json").read_text())[
    "records"
]
selected_final = {row["external_id"] for row in selection if row["split"] == "final_test"}
ids = manifest["primary_ids"]
assert len(ids) == len(set(ids)) == 121
assert set(ids).isdisjoint(manifest["excluded_ids"])
assert set(ids) | set(manifest["excluded_ids"]) == selected_final
refs = {row["external_id"]: row for row in reference["records"]}
assert {row["external_id"] for row in reference["records"] if row["primary_eligible"]} == set(ids)
tags = sorted(topic_tags("v2"))
truth = [refs[i]["tags"] for i in ids]
multilabel_metrics(
    truth, [[] for _ in ids], tags
)  # Validate reference slugs, including three-tag answers.
# Only titles reach model code; references are used later by evaluation.
events = [{"external_id": i, "title": snapshot[i]["title"]} for i in ids]
local = dotenv_values(root / ".env", interpolate=False)
live_value = str(
    os.environ.get("NOTEBOOK10_RUN_LIVE", local.get("NOTEBOOK10_RUN_LIVE") or "false")
).lower()
if live_value not in {"true", "false"}:
    raise ValueError("NOTEBOOK10_RUN_LIVE must be true or false")
RUN_LIVE = live_value == "true"
show_table(
    ["Benchmark fact", "Value"],
    [
        ["Original final groups", len(selected_final)],
        ["Excluded confirmed duplicates", len(manifest["excluded_ids"])],
        ["Primary evaluation groups", len(ids)],
        ["Reference version", reference["version"]],
        ["Rubric version", reference["rubric_version"]],
        ["Live calls enabled", RUN_LIVE],
    ],
)


## 1. Verify the preserved rules and classifier

Reuse the same normalization, positive-match selection and classifier cutoff as the earlier notebooks. First reproduce their saved validation predictions; this checks the implementation before final inference. The fitted vectorizers and logistic regression are loaded from the saved artifact and are never fitted again.

In [ ]:
baseline = contract["candidates"][0]
classical = contract["candidates"][1]
llm = contract["candidates"][2]
compiled = {
    tag: [re.compile(pattern) for pattern in patterns]
    for tag, patterns in baseline["patterns"].items()
}
artifact = joblib.load(root / classical["artifact_path"])
assert artifact["catalog_tags"] == tags
assert float(artifact["recommendation_policy"]["threshold"]) == 0.30


def select_scores(scores, *, threshold=None, include_ties=False):
    """Select at most two tags, optionally preserving exact qualifying ties.

    Regex uses positive binary matches. Logistic regression uses its fixed
    development cutoff. Slug order breaks ties deterministically.
    """
    ranked = sorted(
        (
            (tag, float(value))
            for tag, value in scores.items()
            if (value > 0 if threshold is None else value >= threshold)
        ),
        key=lambda pair: (-pair[1], pair[0]),
    )
    chosen = ranked[:2]
    if include_ties and len(ranked) > 2:
        chosen += [pair for pair in ranked[2:] if pair[1] == chosen[-1][1]]
    return [tag for tag, _ in chosen]


def regex_scores(title):
    """Apply the production-derived title normalization and binary per-tag matches."""
    text = (title or "").replace("ß", "ss").replace("ẞ", "SS")
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii").lower()
    text = re.sub(r"\s+", " ", text)
    return {
        tag: int(any(pattern.search(text) for pattern in patterns))
        for tag, patterns in compiled.items()
    }


def classifier_scores(titles):
    """Transform titles with the saved word/character vocabularies and predict scores."""
    vectors = artifact["vectorizers"]
    features = hstack(
        [vectors["word"].transform(titles), vectors["char"].transform(titles)], format="csr"
    )
    return artifact["model"].predict_proba(features)


baseline_report = json.loads((root / baseline["report_path"]).read_text())
classical_report = json.loads((root / classical["model_report"]).read_text())
for run in baseline_report["runs"]:
    if run["candidate"] == "keywords":
        for row in run["predictions"]:
            assert (
                select_scores(
                    regex_scores(snapshot[row["external_id"]]["title"]),
                    include_ties=run["policy"] == "include_cutoff_ties",
                )
                == row["predicted"]
            )
validation_run = next(
    p
    for p in classical_report["policies"]
    if p["model"] == "word_plus_char" and p["policy"] == "strict_two"
)
validation_titles = [snapshot[row["external_id"]]["title"] for row in validation_run["predictions"]]
validation_scores = classifier_scores(validation_titles)
for saved, scores in zip(validation_run["predictions"], validation_scores, strict=True):
    assert select_scores(dict(zip(tags, scores)), threshold=0.30) == saved["predicted"]
    assert np.allclose(scores, [saved["scores"][t] for t in tags], rtol=1e-10, atol=1e-12)
# Boundary checks protect the matched strict-two and optional tie policy.
assert select_scores({"a": 1, "b": 1, "c": 1}) == ["a", "b"]
assert select_scores({"a": 1, "b": 1, "c": 1}, include_ties=True) == ["a", "b", "c"]
assert select_scores({"a": 0, "b": 0}, include_ties=True) == []
assert select_scores({"a": 0.29, "b": 0.30}, threshold=0.30) == ["b"]
print("Rules and saved classifier reproduce the original validation predictions.")


## 2. Generate and preserve final predictions

The two local candidates produce predictions from titles and save them once. Qwen receives the exact successful title-only request template from DR-47 with only the title replaced. Accepted answers, notes, dates and locations are not sent.

Successful hosted responses and terminal invalid responses are saved. Invalid output counts as a failure with no tags; no prompt repair is allowed on final examples. Transient provider stops pause the batch. An incomplete batch is reported as incomplete and does not produce a headline final score. The global request cap prevents unlimited retries across reruns.

In [ ]:
local_path = folder / "local-predictions.json"
identity = {
    "manifest_sha256": file_sha256(folder / "manifest.json"),
    "candidates_sha256": file_sha256(folder / "candidates.json"),
}
if local_path.exists():
    local_results = json.loads(local_path.read_text())
    assert local_results["identity"] == identity
else:
    local_results = {"identity": identity, "regex": [], "tfidf": []}
    for event in events:
        started = perf_counter()
        scores = regex_scores(event["title"])
        local_results["regex"].append(
            {
                "external_id": event["external_id"],
                "scores": scores,
                "predicted": select_scores(scores),
                "with_ties": select_scores(scores, include_ties=True),
                "seconds": perf_counter() - started,
            }
        )
        started = perf_counter()
        scores = dict(zip(tags, map(float, classifier_scores([event["title"]])[0])))
        local_results["tfidf"].append(
            {
                "external_id": event["external_id"],
                "scores": scores,
                "predicted": select_scores(scores, threshold=0.30),
                "with_ties": select_scores(scores, threshold=0.30, include_ties=True),
                "seconds": perf_counter() - started,
            }
        )
    save_private_json(local_path, local_results)

key = os.environ.get("HETZNER_API_KEY") or local.get("HETZNER_API_KEY")
config = {"endpoint": llm["endpoint"], "model": llm["model"], "key_env": "HETZNER_API_KEY"}
assert config["endpoint"] == "https://inference.hetzner.com/api/v1/chat/completions"
assert manifest["paid_calls_allowed"] is False
hosted = []
for position, event in enumerate(events, 1):
    payload = json.loads(json.dumps(llm["payload_template"]))
    payload["messages"][1]["content"] = json.dumps({"title": event["title"]}, ensure_ascii=False)
    path = folder / "hosted-predictions" / f"{event['external_id']}.json"
    if path.exists():
        answer = json.loads(path.read_text())
        assert answer["payload"] == payload and answer["external_id"] == event["external_id"]
    else:
        starts = len(list((folder / "hosted-attempts").glob("*-started.json")))
        if RUN_LIVE and starts + 3 > manifest["max_hosted_request_starts"]:
            print("Request cap reached. Preserve this run and investigate before resuming.")
            break
        answer = chat_request(
            config,
            payload,
            tags=tags,
            cache_dir=folder / "cache",
            allow_live=RUN_LIVE,
            api_key=key,
            max_retry_wait_seconds=30,
            request_timeout_seconds=60,
        )
        answer["external_id"] = event["external_id"]
        if answer["status"] == "not_run" or answer.get("stop_batch"):
            print(
                f"Paused at {position}/{len(events)}: {answer['status']}. Completed responses are saved."
            )
            break
        save_private_json(path, answer)
    hosted.append(answer)
    if position % 10 == 0 or position == len(events):
        print(f"Preserved {position}/{len(events)} Qwen responses")
complete = len(hosted) == len(events)
print("Hosted responses:", len(hosted), "/", len(events), "| Complete:", complete)


## 3. Final quality and uncertainty

Every eligible group stays in the denominator. Empty predictions and invalid responses reduce coverage and recall. Precision measures the correctness of tags actually returned. Report exact-set match and per-tag support alongside the main measures.

The confidence intervals resample the same event groups with replacement 2,000 times using a fixed seed. They describe sample variability, not uncertainty about incorrect reference labels. Rare categories need particular caution. No results below are used for further tuning.

In [ ]:
metrics = {}
intervals = {}
secondary = {}


def bootstrap_metrics(reference_tags, predicted_tags):
    """Resample whole event groups to estimate percentile intervals for micro metrics."""
    per_event = np.array(
        [
            [len(set(a) & set(b)), len(set(b) - set(a)), len(set(a) - set(b)), bool(b)]
            for a, b in zip(reference_tags, predicted_tags, strict=True)
        ],
        dtype=float,
    )
    rng = np.random.default_rng(manifest["bootstrap_seed"])
    sampled = per_event[
        rng.integers(0, len(per_event), size=(manifest["bootstrap_resamples"], len(per_event)))
    ].sum(axis=1)
    tp, fp, fn, covered = sampled.T
    denominators = {
        "micro_precision": tp + fp,
        "micro_recall": tp + fn,
        "micro_f1": 2 * tp + fp + fn,
        "coverage": np.full(len(tp), len(per_event)),
    }
    numerators = {
        "micro_precision": tp,
        "micro_recall": tp,
        "micro_f1": 2 * tp,
        "coverage": covered,
    }
    out = {}
    for metric, denominator in denominators.items():
        valid = denominator > 0
        values = numerators[metric][valid] / denominator[valid]
        out[metric] = {
            "lower": float(np.quantile(values, 0.025)) if len(values) else None,
            "upper": float(np.quantile(values, 0.975)) if len(values) else None,
            "valid_resamples": int(valid.sum()),
        }
    return out


if complete:
    predicted = {
        "Production regex": [x["predicted"] for x in local_results["regex"]],
        "Word + character TF-IDF": [x["predicted"] for x in local_results["tfidf"]],
        "Qwen3.6-35B-A3B-FP8": [x["predicted"] if x["status"] == "ok" else [] for x in hosted],
    }
    for rows in [local_results["regex"], local_results["tfidf"], hosted]:
        assert [x["external_id"] for x in rows] == ids
    for name, guesses in predicted.items():
        metrics[name] = multilabel_metrics(truth, guesses, tags)
        intervals[name] = bootstrap_metrics(truth, guesses)
    show_table(
        ["Candidate", "Precision", "Recall", "F1", "Coverage", "Exact match"],
        [
            [
                name,
                *[
                    percent(m[k])
                    for k in (
                        "micro_precision",
                        "micro_recall",
                        "micro_f1",
                        "coverage",
                        "exact_match",
                    )
                ],
            ]
            for name, m in metrics.items()
        ],
    )
    show_table(
        ["Candidate", "95% precision interval", "95% recall interval", "95% F1 interval"],
        [
            [
                name,
                *[
                    f"{percent(bounds[k]['lower'])}–{percent(bounds[k]['upper'])}"
                    for k in ("micro_precision", "micro_recall", "micro_f1")
                ],
            ]
            for name, bounds in intervals.items()
        ],
    )
    fig, ax = plt.subplots(figsize=(10, 4))
    names = list(metrics)
    for j, metric in enumerate(["micro_precision", "micro_recall", "micro_f1", "coverage"]):
        ax.bar(
            np.arange(len(names)) + j * 0.18,
            [100 * metrics[name][metric] for name in names],
            width=0.18,
            label=metric.replace("micro_", ""),
        )
    ax.set_xticks(np.arange(len(names)) + 0.27, names)
    ax.set(
        ylim=(0, 105), ylabel="Percent", title=f"Final benchmark: {len(ids)} eligible event groups"
    )
    ax.axhline(80, color="grey", linestyle="--", linewidth=1, label="Precision / recall target")
    ax.axhline(95, color="black", linestyle=":", linewidth=1, label="Coverage target")
    ax.legend()
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    for name, m in metrics.items():
        print(name)
        show_table(
            ["Tag", "Reference count", "Precision", "Recall", "F1"],
            [
                [tag, v["support"], percent(v["precision"]), percent(v["recall"]), percent(v["f1"])]
                for tag, v in m["per_tag"].items()
            ],
        )
    for key in ["regex", "tfidf"]:
        secondary[key] = multilabel_metrics(
            truth, [x["with_ties"] for x in local_results[key]], tags
        )
    show_table(
        [
            "Secondary exact-tie policy",
            "Precision",
            "Recall",
            "F1",
            "Coverage",
            "More than two tags",
        ],
        [
            [
                name,
                *[
                    percent(m[k])
                    for k in ("micro_precision", "micro_recall", "micro_f1", "coverage")
                ],
                m["more_than_two"],
            ]
            for name, m in secondary.items()
        ],
    )
else:
    print("No final comparison yet: complete the preserved hosted batch first.")


## 4. Runtime, failures, cost and recommendation

Saved local timings include title transformation and selection for each event, but exclude loading the artifact. Hosted timings come from the original requests and include retries within that request. Any manual pause between notebook runs is reported separately as an operational limitation. Training cost and local compute cost are not measured here.

The account was authorized as free tier. Token usage is recorded, but invoice charges are unverified: cost cannot be marked as a passed target merely because no invoice was read. The recommendation checks precision ≥80%, recall ≥80% and coverage ≥95%; runtime and cost evidence are reported separately.

In [ ]:
if complete:
    failures = sum(x["status"] != "ok" for x in hosted)
    timing = {}
    for name, values in [
        ("Production regex", [x["seconds"] for x in local_results["regex"]]),
        ("Word + character TF-IDF", [x["seconds"] for x in local_results["tfidf"]]),
        (
            "Qwen3.6-35B-A3B-FP8",
            [x["duration_seconds"] for x in hosted if x.get("duration_seconds") is not None],
        ),
    ]:
        timing[name] = {
            "median_seconds": float(np.median(values)),
            "p95_seconds": float(np.quantile(values, 0.95)),
            "max_seconds": float(max(values)),
            "count": len(values),
        }
    show_table(
        ["Candidate", "Median seconds", "P95 seconds", "Max seconds"],
        [
            [name, *[round(v[k], 4) for k in ("median_seconds", "p95_seconds", "max_seconds")]]
            for name, v in timing.items()
        ],
    )
    eligible = [
        name
        for name, m in metrics.items()
        if (m["micro_precision"] or 0) >= 0.8
        and (m["micro_recall"] or 0) >= 0.8
        and m["coverage"] >= 0.95
    ]
    conclusion = (
        "Quality targets met by: "
        + ", ".join(eligible)
        + ". Confirm runtime/cost evidence before a deployment decision."
        if eligible
        else "No candidate meets all three quality/coverage targets. Keep human approval; do not recommend automatic application from this benchmark."
    )
    print(conclusion)
    print("Hosted failures included as empty predictions:", failures)
    starts = sorted((folder / "hosted-attempts").glob("*-started.json"))
    finished = [
        json.loads(p.read_text())
        for p in sorted((folder / "hosted-attempts").glob("*-finished.json"))
    ]
    usage = {
        k: sum(x.get("usage", {}).get(k, 0) or 0 for x in finished)
        for k in ["prompt_tokens", "completion_tokens", "total_tokens"]
    }
    report = {
        "version": "final-results-v1",
        "manifest_sha256": file_sha256(folder / "manifest.json"),
        "reference_sha256": manifest["reference_sha256"],
        "primary_count": len(ids),
        "excluded_ids": manifest["excluded_ids"],
        "ids": ids,
        "primary_metrics": metrics,
        "bootstrap_95": intervals,
        "secondary_tie_metrics": secondary,
        "timing": timing,
        "predictions": predicted,
        "hosted_failures": failures,
        "request_starts": len(starts),
        "finished_attempts": len(finished),
        "attempt_usage": usage,
        "paid_cost_usd": None,
        "cost_status": "Authorized free tier; invoice unverified; local compute cost unmeasured",
        "conclusion": conclusion,
        "quality_eligible": eligible,
        "limitations": [
            "Focused reference revision, not independent full relabeling",
            "Historical training labels",
            "Known duplicates excluded before predictions; unidentified overlap may remain",
            "Bootstrap does not account for reference error",
            "Manual pauses excluded from per-request timings",
            "Final results must not guide retuning",
        ],
    }
    save_private_json(folder / "results.json", report)
    print("Preserved final results:", folder / "results.json")
else:
    print("Progress preserved; final results file is not created for an incomplete batch.")


## 5. What the final benchmark tells us

| Candidate | Precision | Recall | F1 | Coverage |
|---|---:|---:|---:|---:|
| Production regex | 50.0% | 25.4% | 33.7% | 64.5% |
| Word + character TF-IDF logistic regression | 60.2% | 29.6% | 39.7% | 60.3% |
| Qwen/Qwen3.6-35B-A3B-FP8 — title only | 73.2% | 59.3% | 65.5% | 100.0% |

**Qwen performs best on this benchmark, but misses the agreed quality targets.** Of its 153 suggested tags, 112 match the reference and 41 do not. It misses 77 of the 189 reference tags. It returns something for every event; this does not mean every event is tagged correctly. Its entire tag set matches on 46 of 121 events.

Regex leaves 43 events without suggestions; TF-IDF leaves 48. Keeping extra exact ties raises regex F1 to 35.6%, still below Qwen. TF-IDF's tie view is unchanged. These observations do not justify changing a cutoff using final answers.

Qwen completed 121 requests with no retries or invalid outputs. Its median response time was 1.14 seconds and maximum 4.92 seconds. All returned model IDs match the requested model. Usage was 238,930 input tokens and 2,496 output tokens. The account was authorized as free tier, but invoice cost has not been verified.

**Recommendation:** Keep reviewer approval. Qwen is the strongest candidate for a reviewed suggestion workflow among these three, but this test does not meet the agreed gate for deployment. A service prototype would need an explicit decision to proceed under a revised, reviewed objective. Do not turn this final set into a tuning set and then call another run independent final evidence.

**Limits:** The 95% bootstrap F1 intervals are 27.2–40.6% for regex, 32.7–46.7% for TF-IDF and 59.4–71.6% for Qwen. They describe sampling variability, not label correctness. The reference received five approved additions, not a complete independent second review. Rare tags have little support: education has two examples. One reference has three tags while the primary output policy allows two; its extra tag remains in the recall denominator. The bootstrap coverage interval collapses to 100% for Qwen because every sampled event has a suggestion; this does not guarantee 100% coverage on future events.

**Review:** Run All with `NOTEBOOK10_RUN_LIVE=false`. Check the 121-group count, the table and graph, and this recommendation. The private evidence is `reports/final-benchmark/v1/results.json`; the original review CSV remains unchanged. No further labeling is required for DR-25.
